# References

<a id="H:\advent_of_code\leetcode\maximal_rectangle2.ipynb"></a>

Absolutely. Let me start with the histogram problem, then build up to the full maximal rectangle code line-by-line.

---

## Part 1: Largest Rectangle in Histogram

This is the foundation. A histogram is just an array of bar heights:

```
heights = [2, 1, 5, 6, 2, 3]
```

Visualized:

```
    ██
    ██
    ██  ██
    ██  ██  ██
██  ██  ██  ██  ██
██  ██  ██  ██  ██  ██
2   1   5   6   2   3
```

**Goal**: Find the largest rectangle that can be formed entirely within the bars.

### Key insight

For each bar of height `h`, the largest rectangle that **includes that bar** has:
- Height = `h` (can't be taller than the bar itself)
- Width = extends left until a bar shorter than `h`, extends right until a bar shorter than `h`

So for each bar, you need to find the **nearest smaller bar to the left** and the **nearest smaller bar to the right**.

### Example: bar at index 2 (height = 5)

```
heights = [2, 1, 5, 6, 2, 3]
           0  1  2  3  4  5
```

- Nearest smaller to the left: index 1 (height 1)
- Nearest smaller to the right: index 4 (height 2)
- Width = 4 - 1 - 1 = 2 (columns 2 and 3)
- Area = 5 × 2 = 10

### Example: bar at index 3 (height = 6)

- Nearest smaller to the left: index 1 (height 1) — wait, index 2 has height 5 which is smaller than 6, so index 2
- Nearest smaller to the right: index 4 (height 2)
- Width = 4 - 2 - 1 = 1 (just column 3)
- Area = 6 × 1 = 6

### The stack-based solution (optimal)

You use a **monotonic stack** (increasing heights) to find left and right boundaries in one pass:

```python
def largestRectangleArea(heights):
    stack = []           # stores indices, heights are increasing
    max_area = 0
    n = len(heights)
    
    for i, h in enumerate(heights + [0]):  # append 0 to force cleanup
        while stack and heights[stack[-1]] > h:
            # The bar at stack[-1] just found its right boundary at i
            height = heights[stack.pop()]
            # Left boundary is the new top of stack (or -1 if empty)
            left = stack[-1] if stack else -1
            width = i - left - 1
            max_area = max(max_area, height * width)
        stack.append(i)
    
    return max_area
```

But the maximal rectangle problem uses a **different approach** (the left/right array method) because it needs to **carry boundaries across rows**. The stack method resets each row, which doesn't work when you need to track how boundaries change row by row.

---

## Part 2: The maximal rectangle approach (left/right arrays)

Instead of recomputing from scratch each row, we maintain `left[j]` and `right[j]` arrays that get **restricted** (narrowed) as we go down rows. This is why we use the two-pass method rather than a stack.

---

## Part 3: Full code line-by-line with comments

```python
from typing import List

class Solution:
    def maximalRectangle(self, matrix: List[List[str]]) -> int:
        # --- GUARD CLAUSE ---
        # If the matrix is empty, there are no rectangles
        if not matrix or not matrix[0]:
            return 0

        # --- DIMENSIONS ---
        m = len(matrix)          # number of rows
        n = len(matrix[0])       # number of columns

        # --- THREE ARRAYS (one per column) ---
        # height[j]: consecutive 1s above column j (including current row)
        # left[j]:   leftmost column where the rectangle at j can extend
        # right[j]:  rightmost column + 1 where the rectangle at j can extend
        height = [0] * n
        left   = [0] * n
        right  = [n] * n         # initially, no restriction — boundary is past the last column

        max_area = 0

        # --- PROCESS EACH ROW ---
        for i in range(m):       # i = row index
            row = matrix[i]      # current row as a list of strings

            # ==========================================
            # PASS 1: LEFT-TO-RIGHT
            # Update height[j] and left[j]
            # ==========================================
            curr_left = 0        # tracks where the current run of 1s started in THIS row
                                 # resets to 0 at the start of each row

            for j in range(n):   # j = column index, scanning left to right
                if row[j] == "1":
                    # --- HEIGHT ---
                    # If this cell is "1", the column's height grows by 1
                    # (it's stacked on top of previous rows' 1s)
                    height[j] += 1

                    # --- LEFT BOUNDARY ---
                    # left[j] from the row above is the most restrictive left boundary
                    # from previous rows. curr_left is the left boundary from THIS row.
                    # We take the MAX (rightmost) because the rectangle can't extend
                    # past the more restrictive (closer) boundary.
                    # Example: if row above says left=0 but this row has a zero at col 1,
                    # curr_left=2, so left[j] = max(0, 2) = 2
                    left[j] = max(left[j], curr_left)

                else:  # row[j] == "0"
                    # --- HEIGHT ---
                    # A zero breaks the vertical stack — height resets to 0
                    height[j] = 0

                    # --- LEFT BOUNDARY ---
                    # Reset left[j] to 0 (no restriction from above, since height is 0)
                    left[j] = 0

                    # --- CURR_LEFT ---
                    # The current run of 1s ends at this zero.
                    # The next potential run starts at column j+1
                    curr_left = j + 1

            # ==========================================
            # PASS 2: RIGHT-TO-LEFT
            # Update right[j] and calculate area
            # ==========================================
            curr_right = n       # tracks where the current run of 1s ENDS (exclusive)
                                 # starts at n (past the last column)
                                 # resets to n at the start of each row's right pass

            for j in range(n - 1, -1, -1):  # j = n-1, n-2, ..., 0
                if row[j] == "1":
                    # --- RIGHT BOUNDARY ---
                    # right[j] from the row above is the most restrictive right boundary
                    # from previous rows. curr_right is the right boundary from THIS row.
                    # We take the MIN (leftmost) because the rectangle can't extend
                    # past the more restrictive (closer) boundary.
                    right[j] = min(right[j], curr_right)

                    # --- CALCULATE AREA ---
                    # At this point, we know:
                    #   height[j] = how tall the stack is at column j
                    #   left[j]   = leftmost column the rectangle can extend to
                    #   right[j]  = rightmost column + 1 the rectangle can extend to
                    # Width = right[j] - left[j]
                    # Area = height[j] * width
                    area = height[j] * (right[j] - left[j])
                    max_area = max(max_area, area)

                else:  # row[j] == "0"
                    # --- RIGHT BOUNDARY ---
                    # Reset right[j] to n (no restriction, since height is 0)
                    right[j] = n

                    # --- CURR_RIGHT ---
                    # The current run of 1s ends at this zero.
                    # The boundary moves to this column (j).
                    # Any 1s to the left of this zero will have their right boundary
                    # at most j (they can't extend past this zero).
                    curr_right = j

        return max_area
```

---

## Part 4: Trace through the example

Matrix:
```
Row 0:  1 0 1 0 0
Row 1:  1 0 1 1 1
Row 2:  1 1 1 1 1
Row 3:  1 0 0 1 0
```

### Row 0: `[1, 0, 1, 0, 0]`

**Before row 0**: height=[0,0,0,0,0], left=[0,0,0,0,0], right=[5,5,5,5,5]

**Left pass**:
- curr_left=0
- j=0, val="1": height[0]=1, left[0]=max(0,0)=0
- j=1, val="0": height[1]=0, left[1]=0, curr_left=2
- j=2, val="1": height[2]=1, left[2]=max(0,2)=2
- j=3, val="0": height[3]=0, left[3]=0, curr_left=4
- j=4, val="0": height[4]=0, left[4]=0, curr_left=5

**Right pass**:
- curr_right=5
- j=4, val="0": right[4]=5, curr_right=4
- j=3, val="0": right[3]=5, curr_right=3
- j=2, val="1": right[2]=min(5,3)=3, area=1×(3-2)=1, max=1
- j=1, val="0": right[1]=5, curr_right=1
- j=0, val="1": right[0]=min(5,1)=1, area=1×(1-0)=1, max=1

After row 0: height=[1,0,1,0,0], left=[0,0,2,0,0], right=[1,5,3,5,5]

### Row 1: `[1, 0, 1, 1, 1]`

**Left pass**:
- curr_left=0
- j=0, val="1": height[0]=2, left[0]=max(0,0)=0
- j=1, val="0": height[1]=0, left[1]=0, curr_left=2
- j=2, val="1": height[2]=2, left[2]=max(2,2)=2
- j=3, val="1": height[3]=1, left[3]=max(0,2)=2
- j=4, val="1": height[4]=1, left[4]=max(0,2)=2

**Right pass**:
- curr_right=5
- j=4, val="1": right[4]=min(5,5)=5, area=1×(5-2)=3, max=3
- j=3, val="1": right[3]=min(5,5)=5, area=1×(5-2)=3, max=3
- j=2, val="1": right[2]=min(3,5)=3, area=2×(3-2)=2, max=3
- j=1, val="0": right[1]=5, curr_right=1
- j=0, val="1": right[0]=min(1,1)=1, area=2×(1-0)=2, max=3

After row 1: height=[2,0,2,1,1], left=[0,0,2,2,2], right=[1,5,3,5,5]

### Row 2: `[1, 1, 1, 1, 1]`

**Left pass**:
- curr_left=0
- j=0, val="1": height[0]=3, left[0]=max(0,0)=0
- j=1, val="1": height[1]=1, left[1]=max(0,0)=0
- j=2, val="1": height[2]=3, left[2]=max(2,0)=2
- j=3, val="1": height[3]=2, left[3]=max(2,0)=2
- j=4, val="1": height[4]=2, left[4]=max(2,0)=2

**Right pass**:
- curr_right=5
- j=4, val="1": right[4]=min(5,5)=5, area=2×(5-2)=6, max=6
- j=3, val="1": right[3]=min(5,5)=5, area=2×(5-2)=6, max=6
- j=2, val="1": right[2]=min(3,5)=3, area=3×(3-2)=3, max=6
- j=1, val="1": right[1]=min(5,5)=5, area=1×(5-0)=5, max=6
- j=0, val="1": right[0]=min(1,5)=1, area=3×(1-0)=3, max=6

After row 2: height=[3,1,3,2,2], left=[0,0,2,2,2], right=[1,5,3,5,5]

### Row 3: `[1, 0, 0, 1, 0]`

**Left pass**:
- curr_left=0
- j=0, val="1": height[0]=4, left[0]=max(0,0)=0
- j=1, val="0": height[1]=0, left[1]=0, curr_left=2
- j=2, val="0": height[2]=0, left[2]=0, curr_left=3
- j=3, val="1": height[3]=3, left[3]=max(2,3)=3
- j=4, val="0": height[4]=0, left[4]=0, curr_left=5

**Right pass**:
- curr_right=5
- j=4, val="0": right[4]=5, curr_right=4
- j=3, val="1": right[3]=min(5,4)=4, area=3×(4-3)=3, max=6
- j=2, val="0": right[2]=5, curr_right=2
- j=1, val="0": right[1]=5, curr_right=1
- j=0, val="1": right[0]=min(1,1)=1, area=4×(1-0)=4, max=6

**Final answer: 6** ✓

---

## Summary of the key ideas

1. **Histogram reduction**: Each row is a histogram of heights. The maximal rectangle in the matrix is the maximal rectangle in one of these histograms.

2. **Left/right boundaries**: For each column, the rectangle that uses that column is bounded by the nearest column with smaller height on each side.

3. **Two passes**: Left-to-right finds left boundaries (need to know what came before). Right-to-left finds right boundaries (need to know what comes after).

4. **Carrying across rows**: `left[j]` and `right[j]` persist across rows, getting more restrictive (narrower) as we go down. This is the key difference from the stack-based histogram solution.

5. **Area formula**: `height[j] × (right[j] - left[j])` at each cell, track the maximum.

Would you like me to write this into your notebook? If so, toggle to Act mode.